In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Predicting Total Medical Claims with Machine Learning

In this notebook, I build a machine learning model to predict **total medical claims** for individuals based on their demographics and medical history.

**Goals:**
- Explore the dataset (distributions, missing values, correlations).
- Build baseline models (Linear Regression, Random Forest, etc.).
- Compare model performance.
- Interpret which features are most important for predicting medical costs.

## 📂 Loading the Dataset

The dataset has been added to the Kaggle input directory and contains medical insurance information.  
Below, we load the CSV file and preview the first few rows to ensure it imported correctly.

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/medical-insurance-cost-prediction/medical_insurance.csv")
df.head()

## 🔎 Dataset Overview

After loading the dataset, we begin by examining its structure using `df.info()`.

This allows us to see:

- Number of rows and columns  
- Data types for each feature  
- Whether columns are numerical or categorical  
- Which variables may require preprocessing  

The dataset contains **54 columns** and **100,000 rows**, representing demographic, lifestyle, medical history, and health metric information for each individual.

In [ ]:
df.info()

## 🧭 Checking for Missing Values

Before preprocessing, it is important to determine whether the dataset contains missing or incomplete entries.

Using `df.isnull().sum()`, we identify that almost all columns have **zero missing values**, except for the `alcohol_freq` feature, which contains **30,083 missing entries**.

This step helps guide our cleaning strategy by highlighting which variables require imputation or transformation.

In [ ]:
df.isnull().sum()

## 🧹 Handling Missing Values

The only feature with missing values is **`alcohol_freq`**, which represents how frequently an individual consumes alcohol.

Upon inspection, the column includes categories such as:
- Daily  
- Weekly  
- Occasional  
- None  

However, the value **"None"** is treated as missing in this dataset. To address this issue and avoid losing meaningful information, we:

1. Replace `'None'` with `'Never'` to correctly represent individuals who do not drink.
2. Fill any remaining missing values in the column with `'Never'`.

This ensures that:
- The feature is clean and complete  
- The model interprets nondrinkers correctly  
- We avoid introducing bias by dropping rows

After cleaning, the distribution of the categories becomes more accurate.

In [ ]:
df['alcohol_freq'] = df['alcohol_freq'].replace('None', 'Never')
df['alcohol_freq'].fillna('Never', inplace=True)
df['alcohol_freq'].value_counts()

## 🛠️ Feature Engineering

To improve model performance and capture more meaningful medical patterns, several new features were engineered:

### **1. BMI Category**
BMI values were grouped into clinically relevant categories:
- Underweight  
- Normal  
- Overweight  
- Obese  

This allows models to understand health risk patterns that are not easily seen in raw numerical BMI.

### **2. Chronic Score**
A combined metric created from:
- number of chronic conditions  
- number of medications  

This score reflects overall disease burden.

### **3. Visit Rate**
Calculated as:
`visits_last_year / policy_term_years`  

This represents healthcare utilization frequency.

### **4. Hospital Severity**
Measured using:
`days_hospitalized_last_3yrs / hospitalizations_last_3yrs`  

This captures how severe each hospitalization was on average.

### **5. Procedure Intensity Score**
A sum of all medical procedures:
- imaging  
- surgery  
- lab tests  
- consultations  
- physiotherapy  

This feature reflects overall medical service complexity.

These engineered features help reveal important patterns about patient health levels and cost drivers that raw variables may not capture.

In [ ]:
df['bmi_category'] = pd.cut(df['bmi'],
                            bins=[0, 18.5, 25, 30, 100],
                            labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

df['chronic_score'] = df['chronic_count'] * df['medication_count']

df['visit_rate']= df['visits_last_year']/df['policy_term_years'].replace(0,1)

df['hospital_severity'] = df['days_hospitalized_last_3yrs'] / df['hospitalizations_last_3yrs'].replace(0,1)

df['procedure_intensity_score'] = (
    df['proc_imaging_count'] +
    df['proc_surgery_count'] +
    df['proc_lab_count'] +
    df['proc_consult_count'] +
    df['proc_physio_count']
)
df.head()

# 🔍 Exploratory Data Analysis (EDA)

After cleaning and feature engineering, we explore the dataset to understand how different factors influence annual medical costs.

---

## 📌 1. Target Variable Distribution
We begin by examining the distribution of **annual medical cost** to identify skewness, spread, and outliers.

A log-transform is applied to reduce right skew and make the distribution more suitable for modeling.

---

## 📌 2. Correlation with Numerical Features
Using a correlation matrix, we identify which numerical features show the strongest relationships with annual medical cost.

This helps:
- guide feature selection  
- highlight medically important patterns  
- identify potential multicollinearity  

The heatmap provides a clear visual summary of these relationships.

---

## 📌 3. Medical Cost Across Categories
Boxplots were created for several categorical variables including:
- BMI category  
- Smoker status  
- Education  
- Plan type  
- Network tier  

These comparisons show how different risk groups or insurance tiers affect medical spending.

---

## 📌 4. Key Feature Relationships
Scatterplots help illustrate how certain variables correlate with annual medical cost, including:

- **BMI vs Medical Cost**  
  Shows whether higher BMI leads to higher healthcare expenses.

- **Chronic Count vs Cost**  
  Visualizes the relationship between health risk and spending.

- **Procedure Intensity vs Cost**  
  Indicates how extensive medical procedures influence total cost.

These plots help validate feature engineering choices and reveal patterns that models can learn from.

---

By combining statistical summaries, correlation analysis, and visualizations, the EDA section provides a strong foundation for selecting and training machine learning models.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Overview of Target Variable
print("Summary Statistics for Annual Medical Cost:")
display(df['annual_medical_cost'].describe())

plt.figure(figsize=(7,5))
sns.histplot(df['annual_medical_cost'], bins=50, kde=True)
plt.title('Distribution of Annual Medical Cost')
plt.xlabel('Annual Medical Cost (USD)')
plt.ylabel('Count')
plt.show()

# Log-transform for skewed cost distribution
df['log_medical_cost'] = np.log1p(df['annual_medical_cost'])

plt.figure(figsize=(7,5))
sns.histplot(df['log_medical_cost'], bins=50, kde=True, color='teal')
plt.title('Log-Transformed Distribution of Annual Medical Cost')
plt.xlabel('Log(Annual Medical Cost)')
plt.ylabel('Count')
plt.show()


# Correlation with Numeric Features
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
corr = df[numeric_cols].corr()['annual_medical_cost'].sort_values(ascending=False)

print("\nTop 10 Correlations with Annual Medical Cost:")
display(corr.head(10))

plt.figure(figsize=(7,5))
sns.heatmap(df[numeric_cols].corr(), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Numerical Features)')
plt.show()


# Cost Comparison Across Categories
categorical_features = ['bmi_category', 'smoker', 'education', 'plan_type', 'network_tier']

for col in categorical_features:
    if col in df.columns:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=col, y='annual_medical_cost', data=df)
        plt.title(f'Annual Medical Cost by {col.capitalize()}')
        plt.xticks(rotation=45)
        plt.show()


# Key Feature Relationships
plt.figure(figsize=(7,5))
sns.scatterplot(x='bmi', y='annual_medical_cost', data=df)
plt.title('BMI vs Annual Medical Cost')
plt.show()

plt.figure(figsize=(7,5))
sns.scatterplot(x='chronic_count', y='annual_medical_cost', data=df)
plt.title('Chronic Conditions vs Annual Medical Cost')
plt.show()

plt.figure(figsize=(7,5))
sns.scatterplot(x='procedure_intensity_score', y='annual_medical_cost', data=df)
plt.title('Procedure Intensity vs Annual Medical Cost')
plt.show()

## ✔️ EDA Summary and Key Insights

The exploratory analysis revealed several important patterns in the dataset:

- **Annual medical costs are heavily right-skewed**, indicating that a small number of individuals generate extremely high expenses. A log-transform helps stabilize this distribution.
- Strong correlations were observed between medical cost and several numerical features, including **BMI**, **chronic conditions**, **hospitalization history**, and **procedure counts**.
- Categorical comparisons showed clear cost differences across **smoker status**, **BMI category**, and **insurance plan type**, reaffirming the importance of these features.
- Newly engineered features such as **chronic_score**, **hospital_severity**, and **procedure_intensity_score** showed meaningful relationships with the target variable, suggesting they may improve model performance.

Overall, the EDA confirms that both lifestyle factors and medical history contribute significantly to healthcare spending.  
With a clearer understanding of the data and key predictors, we can now proceed to the **modeling stage** to build and evaluate predictive models.

# ⚙️ Modeling

Based on insights from the EDA phase, we selected a subset of features that showed strong relationships with annual medical cost. These include premium-related variables, medical utilization features, and engineered health-risk indicators.

We begin by preparing the dataset for modeling:

- **X** contains the selected top features  
- **y** contains the target variable (`annual_medical_cost`)
- The data is split into **80% training** and **20% testing** for fair evaluation

Each model is evaluated using:

- **MAE** – Mean Absolute Error  
- **RMSE** – Root Mean Squared Error  
- **R² Score** – Proportion of variance explained by the model

In [ ]:
top_features = [
    'monthly_premium',
    'annual_premium',
    'log_medical_cost',
    'total_claims_paid',
    'avg_claim_amount',
    'risk_score',
    'chronic_count',
    'is_high_risk',
    'days_hospitalized_last_3yrs'
]

X = df[top_features]
y = df['annual_medical_cost']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### **1. Linear Regression**
A baseline model used to understand linear relationships between features and annual medical cost.  
It serves as a benchmark to compare improvements from more advanced models.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression:")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)
print("R2:", lr_r2)

### **2. Random Forest Regressor**
An ensemble method that captures nonlinear interactions and handles complex patterns more effectively than linear models.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest:")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

### **3. Gradient Boosting Regressor**
A boosting model that builds trees sequentially, correcting errors at each step.  
Often effective for structured tabular data with many interacting features.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train, y_train)

gbr_pred = gbr.predict(X_test)

gbr_mae = mean_absolute_error(y_test, gbr_pred)
gbr_rmse = np.sqrt(mean_squared_error(y_test, gbr_pred))
gbr_r2 = r2_score(y_test, gbr_pred)

print("Gradient Boosting:")
print("MAE:", gbr_mae)
print("RMSE:", gbr_rmse)
print("R2:", gbr_r2)

## 📊 Model Performance Comparison

After training all three models, we compare their performance on the test set:

- **Linear Regression** shows reasonable accuracy but struggles with nonlinear relationships.
- **Random Forest** achieves extremely low error and near-perfect R², indicating strong predictive ability.
- **Gradient Boosting** also performs very well, with slightly higher MAE than Random Forest but similar R².

Overall, ensemble models significantly outperform the linear baseline due to their ability to handle complex interactions, skewed distributions, and engineered health-risk features.

A comparison table summarizes the results for clarity.

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Gradient Boosting'],
    'MAE': [lr_mae, rf_mae, gbr_mae],
    'RMSE': [lr_rmse, rf_rmse, gbr_rmse],
    'R2 Score': [lr_r2, rf_r2, gbr_r2]
})

results

# 🔧 Hyperparameter Tuning (Random Forest)

After comparing several baseline models, Random Forest showed the strongest performance.  
To further improve accuracy and reduce prediction error, we apply **RandomizedSearchCV** to tune key hyperparameters:

### Parameters Tuned
- **n_estimators**: number of trees in the forest  
- **max_depth**: controls how deep each tree can grow  
- **min_samples_split**: minimum samples required to split a node  
- **min_samples_leaf**: minimum samples required at a leaf  

RandomizedSearchCV is chosen because:
- It is faster than GridSearchCV  
- It explores a wider search space efficiently  
- It works well for large datasets and complex models  

The tuning process evaluates multiple combinations of parameters using **cross-validation** and selects the configuration that minimizes **Mean Absolute Error (MAE)**.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_rand = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    scoring='neg_mean_absolute_error',
    cv=2,
    n_jobs=-1
)

rf_rand.fit(X_train, y_train)

print("Best params:", rf_rand.best_params_)

## 📈 Tuned Model Performance

After fitting the tuned Random Forest model, we evaluate its performance on the test set.

The tuned model achieves:

- **MAE:** ~0.87  
- **RMSE:** ~45.14  
- **R² Score:** ~0.9998  

Compared to the baseline Random Forest, the tuned version improves stability while maintaining very high accuracy.  
This confirms that the selected hyperparameters enhance the model’s ability to generalize.

In [ ]:
best_rf = rf_rand.best_estimator_
best_rf_pred = best_rf.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Tuned RF MAE:", mean_absolute_error(y_test, best_rf_pred))
print("Tuned RF RMSE:", np.sqrt(mean_squared_error(y_test, best_rf_pred)))
print("Tuned RF R2:", r2_score(y_test, best_rf_pred))

## ⭐ Feature Importance

To understand which features contribute most to predicting annual medical cost, we analyze the feature importance scores from the tuned Random Forest model.

Key insights:

- **log_medical_cost** dominates predictive power, suggesting that log transformation effectively captures underlying cost behavior.
- Premium variables (annual and monthly premium), medical history, and utilization-related features also show meaningful contributions.
- Engineered features such as **procedure intensity score** and **chronic count** contribute smaller but still relevant signals.

Feature importance helps validate modeling decisions and may guide future feature engineering or simplification of the model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(6,4))
plt.title("Feature Importance")
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), X.columns[indices], rotation=45)
plt.show()

## ✔️ Tuning Conclusion

Hyperparameter tuning significantly strengthened the Random Forest model, resulting in lower error and exceptionally high predictive accuracy.  
The improvement demonstrates the value of adjusting model complexity, preventing underfitting or overfitting, and optimizing decision tree behavior.

The tuned Random Forest remains the best performer, offering:
- Strong generalization
- Robust handling of nonlinear patterns
- Interpretability through feature importance

This tuned model will be used as the final model for predicting annual medical cost.

## 📌 Conclusion

This project demonstrated how predictive modeling and health analytics can be used to understand and estimate annual medical insurance costs. By exploring the dataset and evaluating factors such as BMI, smoking status, risk score, and claim history, we identified several key drivers that significantly influence medical expenses. These findings support the idea that health and lifestyle behaviors play a major role in determining healthcare spending and insurance pricing.

We tested multiple regression models, including Linear Regression, Random Forest, and Gradient Boosting. Linear Regression performed the weakest due to the nonlinear nature of medical costs. In contrast, ensemble models—particularly Random Forest—achieved outstanding accuracy and captured complex interactions within the data. Random Forest ultimately provided the best balance of accuracy, interpretability, and stability for predicting annual medical costs.

Overall, this project highlights the value of applying machine learning to insurance analytics. Through feature importance, correlation analysis, and predictive modeling, we uncovered meaningful patterns that explain where medical costs originate and which individuals are most likely to generate higher expenses. These insights can support insurance providers and policymakers in estimating costs, assessing risk, and making more informed decisions about pricing and coverage.


## ⚠️ Limitations

Although the models performed well, several limitations should be acknowledged:

1. **Dataset scope** – The dataset is limited in size and does not include all real-world variables that influence medical spending. Important factors such as detailed medical history, socioeconomic conditions, and policy-specific factors are not represented.

2. **Correlated and missing variables** – Some features may be correlated or influenced by external factors not included in the dataset. This can affect fairness, bias, and overall generalization.

3. **Model sensitivity** – Ensemble models like Random Forest perform strongly but may still be affected by outliers or sampling bias.

4. **Real-world complexity** – Predictive accuracy in a simulation does not fully translate to real insurance cost forecasting. Claims vary by provider, region, coverage, and healthcare access, which this dataset does not capture.

Therefore, results should be interpreted as analytical insights—not as a complete actuarial pricing system.


## 🔮 Future Work

Several enhancements could strengthen this analysis:

1. **Expand the dataset** by adding more features such as procedure details, chronic disease severity, income, lifestyle habits, or insurance policy structure. This would create a more realistic representation of medical spending behavior.

2. **Test advanced models** such as XGBoost, CatBoost, or neural networks, which may capture deeper nonlinear relationships.

3. **Use explainability tools** like SHAP values to better understand how individual features influence predictions, adding interpretability and trust.

4. **Validate the model on real-world insurance data** or across different populations to improve generalizability.

By expanding both the data and modeling techniques, future studies could develop more powerful and reliable tools for insurance pricing, medical risk prediction, and healthcare policy analysis.